# 02 — Factor Data

Loads the seven factors and the zero-beta return series, restricted to the sample window January 2000 – June 2022.

| | |
|---|---|
| **Inputs** | Kenneth French Data Library: `F-F_Research_Data_5_Factors_2x3.csv`, `F-F_Momentum_Factor.csv`; Ľuboš Pástor's website: `liq_data_1962_2025.txt`; supervisor-provided daily zero-beta returns `Zero_beta_returns.csv` |
| **Outputs** | `factors.parquet` (270 months: `mkt_rf smb hml rmw cma rf mom liq_traded`, decimals), `rz_monthly.parquet` (270 months) → both read by NB03 |
| **Thesis** | Sections 3.4–3.5, Table 2 (seven-factor summary statistics) |

**To replicate:** set `DATA_ROOT` in `thesis_config.py` (see README), then *Restart Kernel → Run All*.

**Data vintage.** The French and Pástor files are revised over time, and their header lengths change between vintages.
The thesis uses the files built from the **202604** CRSP database (stated in each French file's header); use the copies
shipped with the replication package rather than a fresh download. Section 2 checks the result against thesis Table 2.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

from thesis_config import DATA_DIR, FACTOR_DIR, FACTORS as FACTOR_COLS, assert_matches, check_paths, save_parquet

FF5_FILE = FACTOR_DIR / "F-F_Research_Data_5_Factors_2x3.csv"
MOM_FILE = FACTOR_DIR / "F-F_Momentum_Factor.csv"
PS_FILE  = FACTOR_DIR / "liq_data_1962_2025.txt"
RZ_FILE  = FACTOR_DIR / "Zero_beta_returns.csv"
check_paths(FF5_FILE, MOM_FILE, PS_FILE, RZ_FILE, DATA_DIR)

SAMPLE_START, SAMPLE_END = "2000-01-01", "2022-06-30"
print(f"pandas {pd.__version__} | numpy {np.__version__}")


def monthly_rows(df):
    """Keep YYYYMM rows only (drops the annual block, headers and copyright lines)."""
    df["date"] = df["date"].astype(str).str.strip()
    return df[(df["date"].str.len() == 6) & df["date"].str.isnumeric()].copy()

## 1. Seven factors

- **FF5** (`Mkt-RF, SMB, HML, RMW, CMA`) and the one-month T-bill rate `RF`: French "Fama/French 5 Factors (2x3)", monthly.
- **MOM**: French "Momentum Factor", monthly (Carhart 1997). Missing values are coded −99.99 / −999.
- **LIQ**: Pástor–Stambaugh (2003) traded liquidity factor, column 4 of Pástor's file (the other two columns are the
  aggregate liquidity level and its innovations, which are not returns). Missing values are coded −99.

French files are in percent and are divided by 100; Pástor's traded factor is already in decimals.

In [ ]:
# ── FF5 + RF ───────────────────────────────────────────────────────────────
ff5 = pd.read_csv(FF5_FILE, skiprows=3, header=0)
ff5.columns = ["date", "mkt_rf", "smb", "hml", "rmw", "cma", "rf"]
ff5 = monthly_rows(ff5)
ff5["date"] = pd.to_datetime(ff5["date"], format="%Y%m")
ff5 = ff5[(ff5["date"] >= SAMPLE_START) & (ff5["date"] <= SAMPLE_END)]
for col in ["mkt_rf", "smb", "hml", "rmw", "cma", "rf"]:
    ff5[col] = pd.to_numeric(ff5[col], errors="coerce") / 100

# ── MOM ────────────────────────────────────────────────────────────────────
mom = pd.read_csv(MOM_FILE, skiprows=12, header=0)
mom.columns = ["date", "mom"]
mom = monthly_rows(mom)
mom["mom"] = pd.to_numeric(mom["mom"], errors="coerce")
mom.loc[mom["mom"].isin([-99.99, -999]), "mom"] = float("nan")
mom["date"] = pd.to_datetime(mom["date"], format="%Y%m")
mom = mom[(mom["date"] >= SAMPLE_START) & (mom["date"] <= SAMPLE_END)]
mom["mom"] = mom["mom"] / 100

# ── Pástor-Stambaugh traded liquidity ─────────────────────────────────────
ps = pd.read_csv(PS_FILE, sep=r"\s+", skiprows=11, header=None,
                 names=["date", "agg_liq", "innov_liq", "liq_traded"])
ps = monthly_rows(ps)
ps["liq_traded"] = pd.to_numeric(ps["liq_traded"], errors="coerce").replace(-99, float("nan"))
ps["date"] = pd.to_datetime(ps["date"], format="%Y%m")
ps = ps[["date", "liq_traded"]].dropna()
ps = ps[(ps["date"] >= SAMPLE_START) & (ps["date"] <= SAMPLE_END)]

# ── Merge (date = first day of month) ──────────────────────────────────────
factors = ff5.merge(mom, on="date", how="inner").merge(ps[["date", "liq_traded"]], on="date", how="inner")

assert len(factors) == 270 and factors.notna().all().all(), "Expected 270 complete months (2000-01 to 2022-06)"
print(f"{len(factors)} months, {factors['date'].min():%Y-%m} to {factors['date'].max():%Y-%m}, no missing values")
save_parquet(factors, DATA_DIR / "factors.parquet")

## 2. Table 2 — seven-factor summary statistics

Monthly returns annualised: mean × 12, standard deviation × √12, Sharpe ratio = annualised mean / annualised SD.
The *t*-statistic tests a zero monthly mean (one-sample *t*-test, 270 months).

In [ ]:
f = factors[FACTOR_COLS]
t_stat, p_val = stats.ttest_1samp(f, 0)
table2 = pd.DataFrame({
    "Mean % p.a.":  f.mean() * 12 * 100,
    "Std % p.a.":   f.std() * np.sqrt(12) * 100,
    "Sharpe":       f.mean() / f.std() * np.sqrt(12),
    "t vs 0":       t_stat,
    "Sig.":         ["**" if p < 0.05 else "*" if p < 0.10 else "" for p in p_val],
    "Worst mo. %":  f.min() * 100,
}, index=FACTOR_COLS)
print(table2.round(2).to_string())
print("\n* p<0.10  ** p<0.05")

# Replication check against thesis Table 2 (values as printed there, 2 decimals)
THESIS_TABLE2 = {   #            mean    std  sharpe     t    worst
    "mkt_rf":     (6.12, 15.79, 0.39, 1.84, -17.20),
    "smb":        (3.13, 10.73, 0.29, 1.38, -15.54),
    "hml":        (2.89, 11.93, 0.24, 1.15, -13.83),
    "rmw":        (5.38, 10.18, 0.53, 2.51, -18.95),
    "cma":        (3.94,  7.33, 0.54, 2.55,  -5.21),
    "mom":        (2.25, 18.09, 0.13, 0.59, -34.34),
    "liq_traded": (4.88, 13.89, 0.35, 1.66, -13.56),
}
num = table2.drop(columns="Sig.")
assert_matches("Table 2", num.values, pd.DataFrame(THESIS_TABLE2, index=num.columns).T.values, 2)

## 3. Zero-beta return series R<sub>Z,t</sub>

Daily returns provided by Prof. Gehde-Trapp (semicolon-separated, dates DD.MM.YYYY, 1993 – Dec 2022).
Cleaning, in order:

1. Strip trailing commas.
2. Values stored in a corrupted scientific format with a comma before the exponent (e.g. `-6,40E+09`) cannot be
   recovered and are set to NaN. **213 of the 5,660 in-sample trading days** are affected, spread over 154 of the 270
   months (at most 4 days in any month).
3. Exact zeros are set to NaN. These are the trailing padding after November 2022; none fall inside the sample window.

Monthly R<sub>Z,t</sub> compounds the remaining daily returns within each calendar month: ∏(1 + r<sub>d</sub>) − 1.

In [ ]:
rz = pd.read_csv(RZ_FILE, sep=";", names=["date", "rz"], skiprows=1)
rz["date"] = pd.to_datetime(rz["date"], format="%d.%m.%Y", errors="coerce")

rz["rz"] = rz["rz"].astype(str).str.rstrip(",").str.strip()
corrupted = rz["rz"].str.contains(r",.*E", regex=True, na=False)
rz.loc[corrupted, "rz"] = float("nan")
rz["rz"] = pd.to_numeric(rz["rz"], errors="coerce")
is_zero = rz["rz"] == 0.0
rz.loc[is_zero, "rz"] = float("nan")

in_sample = (rz["date"] >= SAMPLE_START) & (rz["date"] <= SAMPLE_END)
print(f"In-sample trading days: {in_sample.sum():,}")
print(f"  corrupted values set to NaN: {(corrupted & in_sample).sum()}  "
      f"(months affected: {rz.loc[corrupted & in_sample, 'date'].dt.to_period('M').nunique()})")
print(f"  zeros set to NaN:            {(is_zero & in_sample).sum()}")
print(f"  valid:                       {rz.loc[in_sample, 'rz'].notna().sum():,}")

rz = rz[in_sample].copy()
rz["year_month"] = rz["date"].dt.to_period("M")
rz_monthly = (rz.groupby("year_month")["rz"]
                .apply(lambda x: (1 + x.dropna()).prod() - 1 if x.notna().sum() > 0 else float("nan"))
                .reset_index()
                .rename(columns={"rz": "rz_monthly"}))

assert len(rz_monthly) == 270 and rz_monthly["rz_monthly"].notna().all()
print(rz_monthly["rz_monthly"].describe().round(4).to_string())
save_parquet(rz_monthly, DATA_DIR / "rz_monthly.parquet")

## Appendix — correlation of R<sub>Z,t</sub> with the market

In [ ]:
chk = rz_monthly.merge(factors.assign(year_month=factors["date"].dt.to_period("M"))[["year_month", "mkt_rf"]],
                       on="year_month")
pre = chk["year_month"] < pd.Period("2009-01", "M")
print(f"corr(R_Z, MKT-RF): full {chk['rz_monthly'].corr(chk['mkt_rf']):.3f} | "
      f"pre-2009 {chk[pre]['rz_monthly'].corr(chk[pre]['mkt_rf']):.3f} | "
      f"2009+ {chk[~pre]['rz_monthly'].corr(chk[~pre]['mkt_rf']):.3f}")